# Phase 1 -- Build CSV Manifests (API Only)
Each category has its own cell. Run on separate Colab accounts in parallel.

| Category | CSV | Est. Rows | Est. Time |
|----------|-----|-----------|----------|
| Board Meeting | `board_meetings.csv` | ~180,000 | ~3 hrs |
| Insider Trading | `insider_trading.csv` | ~250,000 | ~4 hrs |
| Dividend | `dividend.csv` | ~30,000 | ~1 hr |
| Bonus / Split | `bonus_splits.csv` | ~2,000 | ~20 min |
| Buyback | `buyback.csv` | ~1,200 | ~20 min |
| Demerger | `demerger.csv` | ~5,000 | ~30 min |

## Cell 1 -- Setup (RUN THIS FIRST on every account)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import requests, os, json, time, random, csv
from datetime import datetime, timedelta
from tqdm.notebook import tqdm

BASE = '/content/drive/MyDrive/ArthLLM-2B/corporate_announcements'
LOGS = f'{BASE}/logs'
os.makedirs(LOGS, exist_ok=True)

RATE_LIMIT     = 3
RETRY_WAIT     = 60
SESSION_BUDGET = 11 * 3600
WINDOW_DAYS    = 15

BSE_API  = 'https://api.bseindia.com/BseIndiaAPI/api/AnnGetData/w'
PDF_LIVE = 'https://www.bseindia.com/xml-data/corpfiling/AttachLive/{}'
PDF_HIST = 'https://www.bseindia.com/xml-data/corpfiling/AttachHis/{}'

START = datetime(2011, 1, 1)
END   = datetime(2026, 5, 8)

CSV_HEADER = ['scrip_code','company_name','category','attachment',
              'pdf_url','pdf_url_hist','filing_date','headline','status']


class BSE:
    def __init__(self):
        self.s = requests.Session()
        self.s.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://www.bseindia.com/corporates/ann.html',
            'Origin': 'https://www.bseindia.com',
        })
        self.last = 0
        self._warm()

    def _warm(self):
        try:
            self.s.get('https://www.bseindia.com', timeout=15)
            time.sleep(1)
            self.s.get('https://www.bseindia.com/corporates/ann.html', timeout=15)
            print('  Session ready.', flush=True)
        except Exception as e:
            print(f'  Warm-up warning: {e}', flush=True)

    def _wait(self):
        gap = time.time() - self.last
        if gap < RATE_LIMIT:
            time.sleep(RATE_LIMIT - gap + random.uniform(0.2, 0.8))
        self.last = time.time()

    def api(self, params):
        self._wait()
        r = self.s.get(BSE_API, params=params, timeout=60)
        if r.status_code in (429, 503):
            time.sleep(60)
            r = self.s.get(BSE_API, params=params, timeout=60)
        r.raise_for_status()
        return r.json()


def make_windows():
    wins, cur = [], START
    while cur < END:
        w_end = min(cur + timedelta(days=WINDOW_DAYS - 1), END)
        wins.append((cur.strftime('%Y%m%d'), w_end.strftime('%Y%m%d')))
        cur = w_end + timedelta(days=1)
    return wins


def build_csv(cat_name, cat_filter):
    """Build CSV manifest for one category. Full resume support."""
    session_start = time.time()
    bse = BSE()

    csv_path = f'{BASE}/{cat_name}.csv'
    prog_path = f'{LOGS}/manifest_progress_{cat_name}.json'

    # Load progress
    done_w = set()
    if os.path.exists(prog_path):
        with open(prog_path) as f:
            done_w = set(json.load(f).get('done_windows', []))

    # Create CSV if new
    if not os.path.exists(csv_path):
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(CSV_HEADER)

    # Load existing attachments for dedup
    seen = set()
    try:
        with open(csv_path, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                if row.get('attachment'): seen.add(row['attachment'])
    except: pass

    windows = make_windows()
    pending = [(f,t) for f,t in windows if f'{f}_{t}' not in done_w]

    print(f'\n{"="*60}')
    print(f'  {cat_name} | {cat_filter}')
    print(f'  Windows: {len(windows)} total, {len(pending)} pending')
    print(f'  Existing: {len(seen)} rows in CSV')
    print(f'{"="*60}')

    total_new = 0

    for wi, (wf, wt) in enumerate(tqdm(pending, desc=cat_name)):
        if time.time() - session_start > SESSION_BUDGET:
            print(f'\n  Session budget reached. Re-run to continue.')
            return total_new

        window_ok = False
        window_new = 0
        total_count = 0
        collected = 0
        page = 1

        while True:
            params = {
                'strCat': cat_filter, 'strPrevDate': wf, 'strScrip': '',
                'strSearch': 'P', 'strToDate': wt, 'strType': 'C',
                'strPageNo': str(page),
            }
            try:
                data = bse.api(params)
                window_ok = True
            except:
                time.sleep(RETRY_WAIT)
                try:
                    data = bse.api(params)
                    window_ok = True
                except:
                    break

            rows = []
            if isinstance(data, dict) and 'Table' in data: rows = data['Table']
            elif isinstance(data, list): rows = data

            if page == 1 and isinstance(data, dict) and 'Table1' in data:
                try: total_count = int(data['Table1'][0]['ROWCNT'])
                except: pass

            if not rows: break
            collected += len(rows)

            batch = []
            for ann in rows:
                att = (ann.get('ATTACHMENTNAME') or '').strip()
                if not att or att in seen: continue
                seen.add(att)
                batch.append([
                    str(ann.get('SCRIP_CD', '')),
                    (ann.get('SLONGNAME') or ann.get('SCRIP_NAME') or '').strip(),
                    cat_name, att,
                    PDF_LIVE.format(att), PDF_HIST.format(att),
                    (ann.get('DT_TM') or '').strip(),
                    (ann.get('NEWSSUB') or '').strip(),
                    'pending'
                ])

            if batch:
                with open(csv_path, 'a', newline='', encoding='utf-8') as f:
                    csv.writer(f).writerows(batch)
                window_new += len(batch)

            if total_count > 0 and collected >= total_count: break
            page += 1

        if window_ok:
            done_w.add(f'{wf}_{wt}')
            with open(prog_path, 'w') as f:
                json.dump({'done_windows': list(done_w), 'updated': datetime.now().isoformat()}, f)

        total_new += window_new
        if (wi+1) % 25 == 0: bse._warm()

    print(f'\n  DONE: {cat_name} -- {total_new} new, {len(seen)} total in CSV')
    return total_new


print('Engine loaded. Run any category cell below.')

---
## Board Meetings (~180K rows, ~3 hrs)

In [ ]:
build_csv('board_meetings', 'Board Meeting')

---
## Insider Trading (~250K rows, ~4 hrs)

In [ ]:
build_csv('insider_trading', 'Insider Trading/SAST')

---
## Dividend (~30K rows, ~1 hr)

In [ ]:
build_csv('dividend', 'Dividend')

---
## Bonus / Splits (~2K rows, ~20 min)

In [ ]:
build_csv('bonus_splits', 'Sub-Division/Splits/Consolidation')

---
## Buyback (~1.2K rows, ~20 min)

In [ ]:
build_csv('buyback', 'Buy Back')

---
## Demerger (~5K rows, ~30 min)

In [ ]:
build_csv('demerger', 'Scheme of Arrangement/Amalgamation')

---
## Check All CSV Stats

In [ ]:
import os, json
from datetime import datetime, timedelta

_BASE = '/content/drive/MyDrive/ArthLLM-2B/corporate_announcements'
_LOGS = f'{_BASE}/logs'
_CATS = ['board_meetings','insider_trading','dividend','bonus_splits','buyback','demerger']
_START, _END, _WDAYS = datetime(2011,1,1), datetime(2026,5,8), 15

nwins, cur = 0, _START
while cur < _END:
    cur += timedelta(days=_WDAYS); nwins += 1

total_rows = 0
print(f'{"Category":20s} {"Windows":>12s} {"CSV Rows":>10s} {"CSV Size":>10s}')
print('-'*56)

for cat in _CATS:
    pf = f'{_LOGS}/manifest_progress_{cat}.json'
    w = 0
    if os.path.exists(pf):
        with open(pf) as f: w = len(json.load(f).get('done_windows', []))
    cp = f'{_BASE}/{cat}.csv'
    rows, sz = 0, 0
    if os.path.exists(cp):
        sz = os.path.getsize(cp)
        with open(cp) as f: rows = sum(1 for _ in f) - 1
    total_rows += rows
    print(f'{cat:20s} {w:>5d}/{nwins:>5d}   {rows:>8,d}   {sz/1e6:>6.1f} MB')

print('-'*56)
print(f'{"TOTAL":20s} {"":>12s} {total_rows:>8,d}')